# Day 3

##  Question: How many machines to serve a 70B model?

This question is really asking: **how much GPU memory is required just to store the model weights**, and then **how many servers (“machines”) you need** given the number of GPUs per server.

For a first-pass estimate we usually ignore KV-cache and other runtime overheads (we’ll add those caveats after the math).

### Step 1: Compute weight memory (FP16)
A model with `P` parameters, stored in FP16, uses:

- `bytes_per_param = 2` bytes (FP16)

So weight size is:

- `weight_bytes = P * 2`

For `P = 70B = 70 x 10^9` parameters:

- `weight_bytes = 70e9 * 2 = 140e9 bytes`

Convert to GB / GiB:

- Decimal GB (common in marketing): `140e9 bytes ≈ 140 GB`
- Binary GiB (what GPUs often report internally): `140e9 / 2^30 ≈ 130.4 GiB`

### Step 2: Minimum number of 80GB GPUs to hold the weights
If a single GPU has about `80 GB` of usable memory, the minimum GPU count is:

- `gpus_needed = ceil(140 / 80) = 2`

So: **you need at least 2 × 80GB GPUs** to fit 70B parameters in FP16 *at minimum*.

> Important: in practice you rarely get to use 100% of advertised memory (CUDA/runtime overhead, memory fragmentation, etc.). This can push the requirement to **2 GPUs with a bit of margin**, or sometimes **3+ GPUs** depending on context length and serving setup.

### Step 3: Turn “GPUs needed” into “machines needed”
“Machines” depends on how many GPUs are in one server.

- If your machine has **1 GPU**: machines = 2
- If your machine has **8 GPUs (e.g., a typical DGX/HGX-style server)**: machines = 1, because you can place/shard the model across **multiple GPUs within the same server** (e.g., tensor parallelism).

So the corrected takeaway is:

- **Minimum GPUs** for FP16 weights: **2**
- **Minimum machines**: **depends on GPUs-per-machine**; commonly **1 machine** if it has multiple 80GB GPUs.

### Step 4 (caveat): KV cache + concurrency often dominate at inference
For real serving, GPU memory is not only weights.

- **KV cache** grows with `sequence_length` and `batch_size` (and also depends on model architecture and precision).
- **Concurrency** (how many requests you batch together) increases KV memory.

This means even if FP16 weights fit on 2 GPUs, you may still need **more GPUs** to support:

- long context windows
- higher throughput / higher concurrent requests
- larger batch sizes

In system design interviews, it’s good to explicitly state: “2 GPUs to fit weights; add extra for KV cache and target throughput.”


## Interview-ready answer (what to say)

If asked “How many machines to serve a 70B model?” I’d say something like:

1. **Weight memory (FP16):** `70B * 2 bytes = ~140 GB` of parameters.
2. **GPU requirement:** with `A100 ~80GB`, `ceil(140/80) = 2` GPUs are the **minimum** to fit the weights (using tensor parallel).
3. **Convert to machines using GPUs per server:**
   - `2 GPUs / machine` would be `ceil(2/2)=1 machine`.
   - `1 GPU / machine` would be `ceil(2/1)=2 machines`.
4. **Add serving headroom:** real serving also needs **KV cache** for attention and some overhead, so to hit latency/throughput targets you often allocate **extra GPUs** (and sometimes use quantization like FP8/INT8/4-bit).


## Practice: Latency vs Throughput trade-offs (LLM serving)

In interviews, “latency vs throughput” is usually about how you batch and parallelize work on GPUs.

- **Latency (p50/p95):** how fast a *single request* finishes (queueing wait + compute time).
- **Throughput:** how many requests/tokens you complete per unit time (e.g., requests/sec or tokens/sec).

### The key trade-off
If you increase batching and/or concurrency to improve GPU utilization:

- **Throughput goes up** because GPUs spend less time idle and you amortize overhead (kernel launches, attention setup, etc.).
- **Latency can go up** because requests may **wait** to be grouped into a batch (and because more load increases queueing).

### Main knobs you can mention
- **Batching strategy**
  - *Smaller batch size / more frequent dispatch:* lower waiting time -> lower latency, potentially lower throughput.
  - *Larger batch size / larger batching window:* higher throughput, but higher per-request latency.
- **Request scheduling**
  - Separate work into *prefill* (prompt processing) and *decode* (token-by-token generation).
  - Use priority policies (e.g., shorter prompts / interactive traffic first) to control tail latency.
- **Concurrency / autoscaling**
  - More concurrent requests increases utilization but also increases queue length.
  - Autoscale based on queue depth or GPU utilization to stay within latency SLOs.
- **Parallelism (multi-GPU / tensor parallel)**
  - Improves capacity (more model shards / more replicas), often helps throughput.
  - Can slightly increase per-request overhead (communication), so latency depends on network + sharding setup.
- **Decoding optimizations**
  - Techniques like speculative decoding, better caching, quantization, and optimized KV-cache management can improve *both* latency and throughput.

### How to answer in 30 seconds
“Latency vs throughput is governed by batching and queueing. Larger batches improve GPU utilization and throughput, but they introduce waiting time and can increase tail latency due to longer queues. In practice we pick a batching window and concurrency level that meets our p95/p99 latency SLO while keeping GPUs sufficiently utilized, and we often treat prefill and decode differently.”

### Practice scenario (try answering out loud)
You serve an LLM with the following constraints:

- Traffic is spiky (interactive users + some longer-form generations).
- Target **p95 latency <= 800 ms** for interactive prompts.
- During peak, GPU utilization drops if you dispatch too aggressively.

Questions:
1. What would you change first: batch size, batching window, or concurrency?
2. How would you treat short interactive prompts vs long prompts?
3. What metrics would you monitor to avoid “high throughput but bad tail latency”?


## DAY 5

### EStimation Drill

#### Q-1: Design the storage for Google Maps' road network?

1. Clarify the Scope
What to say:
	"Before choosing a database, I want to estimate the sheer volume of data we need to store. To do this, I’m going to separate the storage into two distinct buckets: the static road graph (the physical streets, intersections, and speed limits) and the dynamic traffic state (the live speeds and congestion). Does that sound like a reasonable approach?"
2. Estimate the Static Graph (The Geometry)
What to say:
	"Let's start with the static graph. I'll model the road network as a directed multigraph where intersections are nodes and roads are edges.
		Nodes: I'll assume there are roughly 1 billion intersections globally.
		Edges: Assuming an average of 3 to 4 roads connecting to every intersection, that gives us about 4 billion edges.
	Next, I'll estimate the size of a single edge. We need to store an Edge ID, Node A ID, Node B ID, distance, static speed limit, and road type. Let's allocate a generous 100 bytes per edge.
		4 billion edges $\times$ 100 bytes = 400 GB.
	Let's add another 100 GB for the nodes and metadata. This puts our total static graph around 500 GB.
	The Takeaway: 500 GB is surprisingly small. It easily fits on a single modern SSD. This tells me our static storage bottleneck won't be disk capacity, but rather keeping this graph partitioned in RAM across our routing servers for low-latency traversal."
3. Estimate the Dynamic Traffic (The Firehose)
What to say:
	"Now, let's look at the dynamic storage, which will be much heavier. We need to ingest live GPS telemetry to update edge weights.
		Active Users: Google Maps has over a billion users. Let's assume 10% are navigating concurrently during a global peak hour, so 100 million active drivers.
		Ping Rate: If a phone sends a GPS ping every 10 seconds, that’s 10 million pings per second hitting our backend.
		Payload Size: A standard ping containing User ID, Latitude, Longitude, Timestamp, and Velocity is about 50 bytes.
		10 million pings/sec $\times$ 50 bytes = 500 MB/sec of continuous write throughput.
	Over a 24-hour period, that's roughly 43 TB of raw telemetry data per day.
	The Takeaway: While the static graph requires strong consistency to ensure we don't route people through non-existent roads—pointing toward a globally replicated relational DB like Spanner—this 500 MB/s traffic firehose requires a highly scalable, write-heavy NoSQL wide-column store like Bigtable."
	Why this approach works:
	You showed data modeling: You explicitly stated you are viewing the problem as a graph (nodes and edges), which immediately scores points.
	You drove the architecture with numbers: You didn't just guess databases. You proved why Spanner and Bigtable (or their generic equivalents) are needed based on the 500 GB vs. 500 MB/sec calculation.
You identified the real bottleneck: You correctly noted that 500 GB isn't a storage problem, it's a memory/compute problem for the routing algorithms.

---
# API Design Principles

These are **foundational** for any system design interview at Google. Every service you design will expose an API, and interviewers expect you to articulate clean, consistent, production-grade API decisions.

## 1. RESTful API Design: Resource Naming, HTTP Methods, Status Codes

REST (Representational State Transfer) treats everything as a **resource** — a noun, not a verb. Your API models the world as a collection of resources that clients can create, read, update, and delete.

### 1.1 Resource Naming Conventions

**The golden rule: URLs are nouns, HTTP methods are verbs.**

| Pattern | Example | Notes |
|---|---|---|
| Collection | `/users` | Plural nouns for collections |
| Specific resource | `/users/123` | Resource ID in the path |
| Sub-resource | `/users/123/orders` | Nested relationship |
| Specific sub-resource | `/users/123/orders/456` | Drill into nested item |

**Google's AIP (API Improvement Proposals) naming style:**

Google follows a strict `{service}/{version}/{resource}` pattern:
```
https://library.googleapis.com/v1/shelves/shelf1/books/book2
```

**Key rules:**
- **Always plural nouns**: `/users` not `/user`, `/documents` not `/document`
- **Lowercase, hyphen-separated** for multi-word: `/chat-rooms/123` not `/chatRooms/123`
- **No verbs in URLs**: ❌ `/getUser/123` or `/deleteUser/123` — the HTTP method already conveys the action
- **Nest to show ownership**, but don't go deeper than 2-3 levels (flatten if deeper)
- **Use query params for filtering**: `/users?status=active&role=admin`

### 1.2 HTTP Methods (CRUD Mapping)

| Method | Action | Idempotent? | Safe? | Example |
|---|---|---|---|---|
| `GET` | Read / List | ✅ Yes | ✅ Yes | `GET /users/123` → returns user 123 |
| `POST` | Create | ❌ No | ❌ No | `POST /users` with body `{name: "Alice"}` → creates new user |
| `PUT` | Full replace | ✅ Yes | ❌ No | `PUT /users/123` with full body → replaces entire user 123 |
| `PATCH` | Partial update | ✅ Yes* | ❌ No | `PATCH /users/123` with `{email: "new@x.com"}` → updates only email |
| `DELETE` | Remove | ✅ Yes | ❌ No | `DELETE /users/123` → deletes user 123 |

> *PATCH is idempotent if your patch is "set field X to value Y" (which it usually is). It's NOT idempotent if it's "append to list" or "increment counter."

**Concrete example — a YouTube-like video service:**

```
GET    /v1/videos              → List videos (with pagination)
GET    /v1/videos/abc123       → Get video metadata
POST   /v1/videos              → Upload/create a new video
PUT    /v1/videos/abc123       → Replace all metadata for video abc123
PATCH  /v1/videos/abc123       → Update title or description only
DELETE /v1/videos/abc123       → Delete the video

GET    /v1/videos/abc123/comments        → List comments on video
POST   /v1/videos/abc123/comments        → Add a comment
DELETE /v1/videos/abc123/comments/c789   → Delete a specific comment
```

### 1.3 HTTP Status Codes

You don't need to memorize all of them. Know these groups and the key codes within each:

| Range | Meaning | Key Codes |
|---|---|---|
| **2xx** | Success | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | Redirection | `301 Moved Permanently`, `304 Not Modified` |
| **4xx** | Client error | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `409 Conflict`, `429 Too Many Requests` |
| **5xx** | Server error | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable`, `504 Gateway Timeout` |

**When to use which (decision tree):**

```
Request succeeded?
├─ YES → Did we create something?
│        ├─ YES → 201 Created
│        └─ NO  → Is there a response body?
│                 ├─ YES → 200 OK
│                 └─ NO  → 204 No Content
└─ NO  → Is it the client's fault?
         ├─ YES → Is it an auth problem?
         │        ├─ Not logged in     → 401 Unauthorized
         │        ├─ Logged in, no perms → 403 Forbidden
         │        └─ Not auth-related:
         │             ├─ Resource doesn't exist → 404 Not Found
         │             ├─ Conflicting state      → 409 Conflict
         │             ├─ Rate limited           → 429 Too Many Requests
         │             └─ Malformed request      → 400 Bad Request
         └─ NO (server's fault)
              ├─ Unhandled exception → 500 Internal Server Error
              ├─ Downstream is down  → 502 Bad Gateway
              └─ Overloaded/maint.   → 503 Service Unavailable
```

### Interview tip
> "When I design an API, I start by identifying the core resources — the nouns. Then I map CRUD operations to standard HTTP methods. I use plural nouns in URLs, nest sub-resources to show ownership, and return precise status codes so clients can programmatically handle errors without parsing response bodies."

## 2. gRPC: Protobufs, Streaming, and When to Use Over REST

### 2.1 What is gRPC?

gRPC is a **high-performance RPC (Remote Procedure Call) framework** built by Google. Instead of sending JSON over HTTP/1.1, gRPC sends **binary-encoded Protocol Buffers (protobufs)** over **HTTP/2**.

**REST vs gRPC at a glance:**

| Dimension | REST (JSON/HTTP) | gRPC (Protobuf/HTTP2) |
|---|---|---|
| Payload format | JSON (text, human-readable) | Protobuf (binary, compact) |
| Transport | HTTP/1.1 (usually) | HTTP/2 (always) |
| Schema / contract | OpenAPI/Swagger (optional) | `.proto` file (mandatory, strongly typed) |
| Code generation | Optional (Swagger codegen) | Built-in (generates client + server stubs) |
| Streaming | Awkward (SSE, WebSockets) | Native (4 streaming modes) |
| Browser support | Universal | Limited (needs grpc-web proxy) |
| Payload size | Larger (~5-10x) | Smaller (binary encoding) |
| Latency | Higher (text parsing) | Lower (binary + HTTP/2 multiplexing) |

### 2.2 Protocol Buffers (Protobufs)

Protobufs are Google's **language-neutral, platform-neutral** serialization format. You define your data structure once in a `.proto` file, then generate code for any language.

**Example `.proto` file for a video service:**

```protobuf
syntax = "proto3";

package video.v1;

// The Video resource
message Video {
  string video_id = 1;
  string title = 2;
  string description = 3;
  int64 duration_seconds = 4;
  string uploader_id = 5;
  google.protobuf.Timestamp created_at = 6;
  repeated string tags = 7;           // list of tags
  VideoStatus status = 8;
}

enum VideoStatus {
  VIDEO_STATUS_UNSPECIFIED = 0;
  PROCESSING = 1;
  READY = 2;
  FAILED = 3;
}

// Request / Response messages
message GetVideoRequest {
  string video_id = 1;
}

message ListVideosRequest {
  int32 page_size = 1;
  string page_token = 2;      // cursor for pagination
}

message ListVideosResponse {
  repeated Video videos = 1;
  string next_page_token = 2;
}

// The service definition
service VideoService {
  rpc GetVideo(GetVideoRequest) returns (Video);
  rpc ListVideos(ListVideosRequest) returns (ListVideosResponse);
  rpc UploadVideo(stream UploadVideoRequest) returns (Video);              // client streaming
  rpc WatchVideoFeed(WatchFeedRequest) returns (stream Video);             // server streaming
  rpc LiveChat(stream ChatMessage) returns (stream ChatMessage);           // bidirectional
}
```

**Why protobufs matter:**
- **Strong typing** — the schema IS the contract; no ambiguity about field names/types
- **Backward compatibility** — add new fields without breaking old clients (field numbers are stable)
- **Compact** — a protobuf message is typically **5-10x smaller** than its JSON equivalent
- **Fast** — binary serialization/deserialization is orders of magnitude faster than JSON parsing

### 2.3 gRPC Streaming Modes

This is gRPC's killer feature over REST. There are **4 modes**:

| Mode | Client sends | Server sends | Use case |
|---|---|---|---|
| **Unary** | 1 request | 1 response | Simple CRUD (same as REST) |
| **Server streaming** | 1 request | Stream of responses | Real-time feed, log tailing, notifications |
| **Client streaming** | Stream of requests | 1 response | File upload in chunks, sensor data ingestion |
| **Bidirectional streaming** | Stream of requests | Stream of responses | Chat, collaborative editing, gaming |

**Concrete examples:**

**Server streaming** — A stock price ticker:
```
Client sends:   WatchStockRequest { symbol: "GOOG" }
Server streams: StockUpdate { price: 178.23, time: ... }
                StockUpdate { price: 178.45, time: ... }
                StockUpdate { price: 178.31, time: ... }
                ... (continues until client disconnects)
```

**Client streaming** — Uploading a large file in chunks:
```
Client streams: UploadChunk { filename: "video.mp4", chunk: <bytes 0-1MB> }
                UploadChunk { chunk: <bytes 1MB-2MB> }
                UploadChunk { chunk: <bytes 2MB-3MB> }
                ... (client signals end)
Server returns: UploadResponse { file_id: "abc123", size_bytes: 52428800 }
```

**Bidirectional** — Live chat:
```
Client → Server: ChatMessage { user: "Alice", text: "Hello!" }
Server → Client: ChatMessage { user: "Bob", text: "Hi Alice!" }
Client → Server: ChatMessage { user: "Alice", text: "How are you?" }
Server → Client: ChatMessage { user: "Bob", text: "Great, thanks!" }
... (both directions flow independently)
```

### 2.4 When to Use gRPC vs REST

**Use gRPC when:**
- **Internal microservice-to-microservice** communication (the dominant use case at Google)
- You need **low latency and high throughput** (binary is faster than JSON)
- You need **streaming** (real-time feeds, uploads, chat)
- You want a **strongly typed contract** enforced at compile time
- **Polyglot environment** — auto-generate clients in Go, Java, Python, C++ from one `.proto`

**Use REST when:**
- **Public-facing APIs** that browsers/third-party devs consume directly
- You want **human-readable** payloads for debugging (curl + JSON)
- The API is **simple CRUD** with no streaming needs
- **Browser clients** without a grpc-web proxy

**Google's pattern:**
> Internally, almost everything at Google is gRPC (called Stubby internally, gRPC is the open-source version). Externally, Google Cloud APIs expose **both** REST and gRPC — the `.proto` file is the source of truth, and a REST/JSON gateway is auto-generated from it.

### Interview tip
> "For service-to-service communication, I'd use gRPC with protobufs — it gives me a strongly typed contract, compact binary encoding, and native streaming support. For any public-facing or browser-consumed endpoints, I'd put a REST/JSON gateway in front, auto-generated from the same proto definition. This way I get the best of both worlds."

## 3. Pagination: Cursor-Based vs Offset-Based

When an API returns a large collection (e.g., millions of search results, a user's order history), you **must** paginate. Sending everything at once is a non-starter for latency, memory, and bandwidth. The question is **how**.

### 3.1 Offset-Based Pagination

The simplest approach. Client sends `offset` (or `page`) and `limit`.

**Request:**
```
GET /v1/videos?offset=40&limit=20
```

**Server logic (SQL):**
```sql
SELECT * FROM videos ORDER BY created_at DESC LIMIT 20 OFFSET 40;
```

**Response:**
```json
{
  "videos": [ ... 20 videos ... ],
  "total_count": 15230,
  "offset": 40,
  "limit": 20
}
```

**Pros:**
- Dead simple to implement
- Client can jump to any page: "give me page 5" → `offset=80, limit=20`
- `total_count` lets you show "Page 3 of 762"

**Cons (and they're serious):**

| Problem | Explanation |
|---|---|
| **O(offset) performance** | `OFFSET 1000000` still scans and discards 1M rows. At scale, deep pages are **brutally slow**. |
| **Inconsistent results on mutation** | If a new item is inserted at the top while the client is paginating, items shift. Page 2 may repeat an item from page 1, or skip one entirely. |
| **Doesn't work with distributed stores** | In systems like Bigtable/Spanner, there's no global row index; offset is meaningless or very expensive. |

### 3.2 Cursor-Based Pagination (Token-Based)

Instead of "skip N rows", the server gives the client an **opaque cursor** (also called `page_token`) that encodes "where you left off." The client sends it back to get the next page.

**Request (first page):**
```
GET /v1/videos?page_size=20
```

**Response:**
```json
{
  "videos": [ ... 20 videos ... ],
  "next_page_token": "eyJ0IjoiMjAyNS0wNC0xMFQxMjowMDowMFoiLCJpZCI6InZpZDk4NyJ9"
}
```

**Request (next page):**
```
GET /v1/videos?page_size=20&page_token=eyJ0IjoiMjAyNS0wNC0xMFQxMjowMDowMFoiLCJpZCI6InZpZDk4NyJ9
```

**What's inside the cursor?** It's typically a base64-encoded JSON or protobuf containing the **sort key values of the last item** on the previous page:
```json
// Decoded cursor:
{ "created_at": "2025-04-10T12:00:00Z", "id": "vid987" }
```

**Server logic (SQL):**
```sql
-- Instead of OFFSET, use a WHERE clause on the sort key:
SELECT * FROM videos
WHERE (created_at, id) < ('2025-04-10T12:00:00Z', 'vid987')
ORDER BY created_at DESC, id DESC
LIMIT 20;
```

This is a **keyset seek** — it uses an index, starts exactly where the previous page ended, and is **O(page_size)** regardless of how deep you are.

**Pros:**

| Advantage | Why |
|---|---|
| **O(page_size) always** | Uses an index seek, never scans skipped rows. Page 50,000 is as fast as page 1. |
| **Consistent under mutations** | Cursor points to a specific item, not a position. Inserts/deletes don't cause duplicates or gaps. |
| **Works with distributed stores** | Bigtable, Spanner, DynamoDB all support range scans from a key. |

**Cons:**
- **No random page access**: Can't "jump to page 50" — must traverse sequentially
- **Cursor is opaque**: Client can't construct or modify it (this is by design — the server controls the format)

### 3.3 Why Google Uses Cursor-Based

All Google Cloud APIs use cursor-based pagination via `page_token` / `next_page_token`. The reasons align perfectly with their infrastructure:

1. **Spanner / Bigtable** — Google's primary databases don't support efficient OFFSET; they're built for range scans on sorted keys
2. **Consistency** — Google's APIs serve constantly-changing datasets; offset would give inconsistent results
3. **Performance at scale** — Billions of rows; offset pagination would be catastrophically slow on deep pages
4. **Standard pattern** — Defined in [Google AIP-158](https://google.aip.dev/158); every Google API follows it

### 3.4 Side-by-Side Comparison

| Aspect | Offset-Based | Cursor-Based |
|---|---|---|
| Performance on page N | O(offset + limit) | O(page_size) |
| Deep pagination | Degrades badly | Constant time |
| Random page access | ✅ Yes | ❌ No |
| Stable under writes | ❌ No (items shift) | ✅ Yes |
| Implementation | Simple | Moderate |
| Distributed DB friendly | ❌ Poor | ✅ Excellent |
| **Google's choice** | ❌ Not used | ✅ Standard |

### Interview tip
> "I'd use cursor-based pagination with an opaque `page_token`. The server encodes the last-seen sort key into the token, and on the next request, does an index seek from that key. This gives O(page_size) performance regardless of depth, handles concurrent mutations gracefully, and works naturally with distributed stores like Spanner. This is exactly the pattern Google uses across all its APIs (AIP-158)."

## 4. Idempotency: Why It Matters and How to Implement It

### 4.1 What Is Idempotency?

An operation is **idempotent** if performing it **once** produces the **same result** as performing it **multiple times**.

```
f(x) = f(f(x))       ← mathematical definition
```

**In API terms:** If a client sends the exact same request twice (intentionally or due to a retry), the system state should be the same as if they sent it once.

**Which HTTP methods are naturally idempotent?**

| Method | Idempotent? | Why |
|---|---|---|
| `GET` | ✅ | Reading never changes state |
| `PUT` | ✅ | "Set X to 5" — doing it twice still leaves X = 5 |
| `DELETE` | ✅ | "Delete item 123" — second call finds nothing to delete, state unchanged |
| `PATCH` | ✅* | "Set email to Y" is idempotent; "append to list" is NOT |
| `POST` | ❌ | "Create a new order" — doing it twice creates TWO orders |

**The dangerous one is `POST`** — and that's exactly where you need to add idempotency.

### 4.2 Why It Matters (The Real-World Horror Story)

Imagine a payment API:

```
POST /v1/payments
{ "from": "alice", "to": "bob", "amount": 100.00 }
```

**What happens without idempotency:**

```
1. Client sends POST /v1/payments → Server processes it, charges Alice $100
2. Network hiccup — client never receives the 200 OK response
3. Client retries POST /v1/payments → Server processes it AGAIN, charges Alice $100
4. Alice just paid $200 instead of $100 💸
```

This isn't hypothetical. This happens constantly in distributed systems because:
- **Networks are unreliable** — timeouts, dropped connections, load balancer resets
- **Clients retry aggressively** — mobile apps, SDKs, and libraries have built-in retry logic
- **At-least-once delivery** — most message queues (Pub/Sub, Kafka) can deliver duplicates

### 4.3 How to Implement Idempotency

**The standard pattern: Idempotency Keys**

The client generates a **unique key** (usually a UUID) and sends it with the request. The server uses this key to detect and deduplicate retries.

**Step-by-step flow:**

```
┌──────────┐                          ┌──────────┐
│  Client   │                          │  Server   │
└─────┬─────┘                          └─────┬─────┘
      │                                      │
      │  POST /v1/payments                   │
      │  Idempotency-Key: "uuid-abc-123"     │
      │  Body: { amount: 100 }               │
      ├─────────────────────────────────────→ │
      │                                      │
      │        1. Check: has "uuid-abc-123"   │
      │           been seen before?           │
      │        2. NO → process payment        │
      │        3. Store key + result in DB    │
      │                                      │
      │  200 OK { payment_id: "pay_456" }    │
      │ ←────────────────────────────────────┤
      │                                      │
      │  ⚡ Network timeout, client retries:  │
      │                                      │
      │  POST /v1/payments                   │
      │  Idempotency-Key: "uuid-abc-123"     │
      │  Body: { amount: 100 }               │
      ├─────────────────────────────────────→ │
      │                                      │
      │        1. Check: has "uuid-abc-123"   │
      │           been seen before?           │
      │        2. YES → return stored result  │
      │        3. Do NOT process again        │
      │                                      │
      │  200 OK { payment_id: "pay_456" }    │
      │ ←────────────────────────────────────┤
```

**Server-side implementation (pseudocode):**

```python
def create_payment(request):
    idempotency_key = request.headers["Idempotency-Key"]

    # Step 1: Check if we've seen this key
    existing = db.get("idempotency_keys", idempotency_key)
    if existing:
        return existing.stored_response   # return cached result, don't re-process

    # Step 2: Process the payment (the actual business logic)
    result = payment_service.charge(request.body)

    # Step 3: Store the key + result atomically
    db.put("idempotency_keys", idempotency_key, {
        "stored_response": result,
        "created_at": now(),
        "expires_at": now() + timedelta(hours=24)   # TTL for cleanup
    })

    return result
```

**Critical implementation details:**

| Detail | Why |
|---|---|
| **Store key BEFORE or atomically with processing** | Otherwise a crash between "process" and "store key" means the retry won't be caught |
| **Use a TTL** (e.g., 24h) | Don't store idempotency keys forever; clients can retry for a window, then it expires |
| **Key must be client-generated** | Server can't know if two requests are "the same" just from the body (same amount to same person could be intentional) |
| **Key scope** | Usually scoped to a user/account — different users can use the same key without collision |

### 4.4 Where Google Uses Idempotency

- **Google Cloud APIs** — Many mutating operations accept a `request_id` field for idempotency (see [AIP-155](https://google.aip.dev/155))
- **Pub/Sub** — Messages have a `message_id`; subscribers use it to deduplicate
- **Spanner** — Mutations can be made idempotent using transaction replay detection
- **Stripe** (industry standard) — Popularized the `Idempotency-Key` header pattern

### 4.5 Beyond API-Level: Idempotent Consumers

The same principle applies to **message consumers** (Pub/Sub, Kafka):

```
Message arrives: { order_id: "ord_789", action: "ship" }

Consumer logic:
  1. Check: is order "ord_789" already shipped? (read from DB)
  2. YES → skip (idempotent — already done)
  3. NO  → ship order, mark as shipped
```

This is called **"idempotent consumer"** or **"deduplication at the consumer"** — essential in any at-least-once delivery system.

### Interview tip
> "Any mutating endpoint — especially payments, order creation, or state transitions — must be idempotent. I'd require clients to send an `Idempotency-Key` header with a UUID. The server checks a key-value store (with a TTL) before processing: if the key exists, return the cached response; if not, process and store atomically. This makes retries safe and is critical in distributed systems where network failures cause duplicate requests."

## 5. Rate Limiting at the API Layer

### 5.1 What Is Rate Limiting and Why?

Rate limiting **restricts how many requests** a client can make to your API within a time window. It's a **protective mechanism** — without it, a single misbehaving client (or attacker) can take down your entire service.

**Why you need it:**

| Threat | Without Rate Limiting | With Rate Limiting |
|---|---|---|
| **Abusive client / bot** | Hammers your API, starves other users | Capped; excess requests get `429 Too Many Requests` |
| **DDoS attack** | Backend overwhelmed, cascading failure | Traffic shedded at the edge before hitting backends |
| **Buggy client SDK** | Retry loop sends 1000 req/sec | Limited to configured quota |
| **Cost control** | Cloud bill spikes unpredictably | Usage bounded to plan limits |
| **Fair sharing** | One tenant monopolizes shared resources | Each tenant gets a fair quota |

### 5.2 Rate Limiting Algorithms

#### Algorithm 1: Fixed Window Counter

Divide time into fixed windows (e.g., 1-minute intervals). Count requests per window.

```
Window: 14:00:00 – 14:00:59 → Counter = 0
Request at 14:00:03 → Counter = 1  ✅
Request at 14:00:15 → Counter = 2  ✅
...
Request at 14:00:47 → Counter = 100 ✅ (limit = 100)
Request at 14:00:48 → Counter would be 101 → ❌ 429 rejected
Window resets at 14:01:00 → Counter = 0
```

**Pros:** Simple, O(1) memory per client.
**Cons:** **Boundary burst problem** — a client can send 100 requests at 14:00:59 and 100 more at 14:01:00 (200 in 2 seconds, despite a "100/min" limit).

#### Algorithm 2: Sliding Window Log

Store the **timestamp of every request**. When a new request arrives, count how many timestamps fall within the last N seconds.

```
Limit: 5 requests per 60 seconds

Timestamps: [14:00:05, 14:00:20, 14:00:35, 14:00:50, 14:01:02]
New request at 14:01:10:
  → Remove timestamps older than 14:00:10 (60s ago)
  → Remaining: [14:00:20, 14:00:35, 14:00:50, 14:01:02] = 4
  → 4 < 5 → ✅ allowed, add 14:01:10
```

**Pros:** Perfectly accurate, no boundary burst issue.
**Cons:** O(n) memory per client (must store every timestamp). Expensive at high request rates.

#### Algorithm 3: Sliding Window Counter (Hybrid)

Combines fixed window's efficiency with sliding window's accuracy. Weight the previous window's count by how much it overlaps with the current sliding window.

```
Limit: 100 req/min

Previous window (14:00–14:01): 80 requests
Current window  (14:01–14:02): 30 requests so far
Current time: 14:01:15 → 25% into current window → 75% of prev window overlaps

Weighted count = (0.75 × 80) + 30 = 60 + 30 = 90  → ✅ under 100
```

**Pros:** O(1) memory, good accuracy.  **Cons:** Approximate (but good enough in practice).

#### Algorithm 4: Token Bucket (Most Common in Production)

A "bucket" holds tokens. Each request consumes a token. Tokens are added at a fixed rate. If the bucket is empty, the request is rejected.

```
Config: bucket_size = 10, refill_rate = 2 tokens/sec

t=0:  bucket = 10 tokens
t=0:  5 requests arrive → bucket = 5    ✅ all allowed
t=1:  +2 tokens refilled → bucket = 7
t=1:  3 requests arrive → bucket = 4    ✅ all allowed
t=2:  +2 tokens refilled → bucket = 6
t=2:  8 requests arrive → 6 allowed, 2 rejected ❌  → bucket = 0
t=3:  +2 tokens refilled → bucket = 2
```

**Pros:** Allows **short bursts** (up to bucket_size) while enforcing an average rate. Simple, efficient, widely used.
**Cons:** Choosing bucket_size and refill_rate requires tuning.

**This is what Google, AWS, and most production systems use.**

#### Algorithm 5: Leaky Bucket

Requests enter a FIFO queue (the "bucket") that is drained at a constant rate. If the queue is full, new requests are dropped.

```
Config: queue_size = 5, drain_rate = 1 req/sec

Burst of 8 requests arrives:
  → 5 queued, 3 dropped immediately ❌
  → Queue drains: 1 req/sec processed
  → Smooth, constant output rate
```

**Pros:** Produces a perfectly smooth output rate (great for downstream services that can't handle bursts).
**Cons:** Adds queuing latency; doesn't handle legitimate bursts well.

### 5.3 Where to Enforce Rate Limits

```
Client → [CDN/Edge] → [API Gateway] → [Load Balancer] → [Service]
              ↑              ↑                               ↑
         Layer 1         Layer 2                         Layer 3
      (coarse DDoS)   (per-user/key)              (per-endpoint fine-grained)
```

| Layer | What it does | Example |
|---|---|---|
| **Edge / CDN** | Drops obvious abuse (IP-level, geo-blocking) | Cloudflare, Google Cloud Armor |
| **API Gateway** | Per-API-key or per-user quotas | Google Cloud Endpoints, Apigee, Kong |
| **Service level** | Fine-grained per-endpoint limits | Custom middleware in your service |

**Google's approach:** Rate limiting happens at the **API Gateway** level (Google Front End / GFE), configured via **quota policies**. Each API key gets a quota (e.g., "100 QPS for the free tier, 10,000 QPS for enterprise"). Quotas are tracked in a distributed counter service.

### 5.4 Distributed Rate Limiting

On a single server, rate limiting is trivial (in-memory counter). The hard part is doing it **across many servers**.

**Challenge:** Client sends requests that hit different backend instances. Each instance only sees a fraction of the traffic.

**Solutions:**

| Approach | How | Trade-off |
|---|---|---|
| **Centralized counter (Redis)** | All instances increment a shared Redis counter | Accurate, but adds a Redis RTT to every request (~1ms) |
| **Local + sync** | Each instance tracks locally, periodically syncs to a central store | Lower latency, but briefly inaccurate during sync gaps |
| **Sticky sessions** | Route all requests from one client to one instance | Simple, but breaks if that instance goes down |
| **Cell-based** | Shard users across cells; each cell does local rate limiting | Google's approach — each cell manages its own quota slice |

**Redis-based implementation (most common):**

```python
import redis
import time

r = redis.Redis()

def is_rate_limited(user_id, limit=100, window=60):
    key = f"rate:{user_id}:{int(time.time()) // window}"

    current = r.incr(key)

    if current == 1:
        r.expire(key, window)  # auto-cleanup after window

    return current > limit
```

### 5.5 Response Headers for Rate Limiting

Good APIs tell clients about their quota status via headers:

```http
HTTP/1.1 200 OK
X-RateLimit-Limit: 100          ← max requests per window
X-RateLimit-Remaining: 42       ← requests left in current window
X-RateLimit-Reset: 1712944800   ← Unix timestamp when window resets

---

HTTP/1.1 429 Too Many Requests
Retry-After: 23                 ← seconds to wait before retrying
X-RateLimit-Limit: 100
X-RateLimit-Remaining: 0
```

This lets well-behaved clients implement **client-side backoff** instead of blindly retrying.

### 5.6 Rate Limiting vs Throttling vs Backpressure

| Concept | What it does | Who does it |
|---|---|---|
| **Rate limiting** | Rejects excess requests outright (429) | Server, at the API gateway |
| **Throttling** | Slows down (queues) excess requests instead of rejecting | Server, with leaky bucket |
| **Backpressure** | Upstream slows down because downstream signals it's overwhelmed | System-wide (e.g., TCP flow control, gRPC flow control) |

### Interview tip
> "I'd implement rate limiting at the API gateway layer using a token bucket algorithm — it allows controlled bursts while enforcing an average rate. For distributed rate limiting across multiple instances, I'd use a centralized Redis counter with a sliding window. The API returns `X-RateLimit-Remaining` headers so clients can self-throttle, and `429 + Retry-After` when limits are exceeded. At the edge, I'd also have coarse IP-level DDoS protection via something like Cloud Armor."